## A2A Registry with Azure CosmosDB 

### Installing Utilities and Libraries

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv

### Setting up the Environment Variables

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv("COSMOSDB_ENDPOINT")
key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient

client = CosmosClient(endpoint, key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)
container = database.get_container_client(container_name)

### Load the JSON Registry

In [ ]:
import json

with open(
    "agents.json",
    "r"
) as file:

    agent_definitions = (
        json.load(file)
    )


print(
    f"Loaded "
    f"{len(agent_definitions)} "
    f"agent definitions."
)

### Convert JSON Definitions into Registry Documents

In [ ]:
from datetime import (
    datetime,
    timedelta,
    timezone
)


def build_registry_document(
    agent
):

    return {

        # Cosmos document ID
        "id":
            agent["agentId"],


        "agentId":
            agent["agentId"],


        "name":
            agent["name"],


        # Partition key
        "category":
            agent["category"],


        "endpointUrl":
            agent["endpointUrl"],


        "capabilities":
            agent["capabilities"],


        "version":
            agent["version"],


        # Runtime metadata
        "healthStatus":
            "healthy",


        "lastHeartbeat":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

### Define the Agent Registration Helper Function

In [ ]:
def register_agent(
    agent
):

    registry_document = (
        build_registry_document(
            agent
        )
    )


    container.upsert_item(
        registry_document
    )


    print(
        "✓",
        agent["category"],
        "→",
        agent["agentId"]
    )

### Register All the Agents in the CosmosDB Registry

In [ ]:
for agent in agent_definitions:

    register_agent(
        agent
    )

### Query Agents by Category

In [ ]:
def get_agents_by_category(
    category
):

    query = """
    SELECT
        c.agentId,
        c.name,
        c.endpointUrl,
        c.capabilities,
        c.version,
        c.healthStatus

    FROM c

    WHERE
        c.category = @category
    """


    parameters = [

        {
            "name":
                "@category",

            "value":
                category
        }
    ]


    return list(

        container.query_items(

            query=query,

            parameters=parameters,

            partition_key=category
        )
    )

In [ ]:
document_agents = (
    get_agents_by_category(
        "document"
    )
)


for agent in document_agents:

    print(
        "\n",
        agent["name"]
    )

    print(
        "agent endpointUrl:",
        agent["endpointUrl"]
    )

    print(
        "agent capabilities:",
        agent["capabilities"]
    )

In [ ]:
research_agents = (
    get_agents_by_category(
        "research"
    )
)


for agent in research_agents:

    print(
        "\n",
        agent["agentId"]
    )
    print(
        "agent endpointUrl:",
        agent["endpointUrl"]
    )
    print(
        "agent capabilities:",
        agent["capabilities"]
    )

### Discover Agents by Category and Capabilities

In [ ]:
def discover_agents(
    category,
    capability
):

    query = """
    SELECT
        c.agentId,
        c.name,
        c.category,
        c.endpointUrl,
        c.capabilities,
        c.version,
        c.healthStatus,
        c.lastHeartbeat

    FROM c

    WHERE
        c.category = @category

    AND
        ARRAY_CONTAINS(
            c.capabilities,
            @capability
        )

    AND
        c.healthStatus = "healthy"
    """

    parameters = [
        {
            "name": "@category",
            "value": category
        },
        {
            "name": "@capability",
            "value": capability
        }
    ]

    return list(
        container.query_items(
            query=query,
            parameters=parameters,
            partition_key=category
        )
    )

In [ ]:
agents = discover_agents(
    category="document",
    capability="invoice-extraction"
)


for agent in agents:

    print(
        "\n",
        "Agent:",
        agent["name"]
    )

    print(
        "Category:",
        agent["category"]
    )

    print(
        "Endpoint:",
        agent["endpointUrl"]
    )

    print(
        "Version:",
        agent["version"]
    )

    print(
        "Capabilities:",
        agent["capabilities"]
    )